In [1]:
import numpy as np
import os
import pickle
import warnings
from PIL import Image
from scipy.ndimage import gaussian_filter
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray

warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
BASE_DIR   = 'img_data'
TRAIN_DIR  = os.path.join(BASE_DIR, 'seg_train')
TEST_DIR   = os.path.join(BASE_DIR, 'seg_test')
FEAT_DIR   = 'features'
os.makedirs(FEAT_DIR, exist_ok=True)

CLASSES      = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IMG_SIZE     = (150, 150)

# Corruption levels — index 0 is always clean
BLUR_SIGMAS        = [0, 1, 2, 3, 4]
BRIGHTNESS_FACTORS = [1.0, 0.8, 0.6, 0.4, 0.2]

In [3]:
def load_images(base_dir):
    images, labels = [], []
    for cls in CLASSES:
        cls_dir = os.path.join(base_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
            img = img.resize(IMG_SIZE, Image.BILINEAR)
            images.append(np.array(img, dtype=np.float32) / 255.0)
            labels.append(CLASS_TO_IDX[cls])
    return np.array(images), np.array(labels)

print('Loading train...')
X_train_raw, y_train = load_images(TRAIN_DIR)
print(f'  {X_train_raw.shape}, labels: {y_train.shape}')

print('Loading test...')
X_test_raw, y_test = load_images(TEST_DIR)
print(f'  {X_test_raw.shape}, labels: {y_test.shape}')

Loading train...
  (14034, 150, 150, 3), labels: (14034,)
Loading test...
  (3000, 150, 150, 3), labels: (3000,)


In [4]:
def colour_histogram(img, n_bins=32):
    """Per-channel colour histogram normalised to sum=1. Shape: (n_bins*3,)"""
    feats = []
    for ch in range(3):
        hist, _ = np.histogram(img[:, :, ch], bins=n_bins, range=(0.0, 1.0))
        feats.append(hist / (hist.sum() + 1e-8))
    return np.concatenate(feats).astype(np.float32)


def hog_features(img):
    """HOG on grayscale image, pixels_per_cell=15. Shape: (2916,)"""
    gray = rgb2gray(img)
    return hog(gray, orientations=9, pixels_per_cell=(15, 15),
               cells_per_block=(2, 2), visualize=False).astype(np.float32)


def lbp_features(img, n_points=24, radius=3):
    """Uniform LBP histogram on grayscale image. Shape: (26,)"""
    gray = rgb2gray(img)
    lbp  = local_binary_pattern(gray, n_points, radius, method='uniform')
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins))
    return (hist / (hist.sum() + 1e-8)).astype(np.float32)


def spatial_features(img, grid=2):
    """
    Divide into grid×grid regions; compute colour+HOG+LBP per region.
    Preserves spatial layout that global histograms discard.
    """
    H, W  = img.shape[:2]
    ch, cw = H // grid, W // grid
    feats  = []
    for i in range(grid):
        for j in range(grid):
            cell = img[i*ch:(i+1)*ch, j*cw:(j+1)*cw]
            feats.append(colour_histogram(cell, n_bins=16))   # 48
            feats.append(hog_features(cell))                  # varies by cell size
            feats.append(lbp_features(cell))                  # 26
    return np.concatenate(feats).astype(np.float32)


def extract_features(img):
    return {
        'colour':  colour_histogram(img),
        'hog':     hog_features(img),
        'lbp':     lbp_features(img),
        'spatial': spatial_features(img),
    }


def extract_batch(images, desc=''):
    result = {k: [] for k in ('colour', 'hog', 'lbp', 'spatial')}
    n = len(images)
    for i, img in enumerate(images):
        if i % 1000 == 0:
            print(f'  {desc}: {i}/{n}')
        f = extract_features(img)
        for k in result:
            result[k].append(f[k])
    return {k: np.array(v) for k, v in result.items()}


# Sanity-check feature shapes on one image
sample = extract_features(X_train_raw[0])
for k, v in sample.items():
    print(f'{k:10s}: {v.shape}')

colour    : (96,)
hog       : (2916,)
lbp       : (26,)
spatial   : (2600,)


In [5]:
def apply_blur(img, sigma):
    if sigma == 0:
        return img.copy()
    out = np.stack(
        [gaussian_filter(img[:, :, c], sigma=sigma) for c in range(3)],
        axis=-1
    )
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def apply_brightness(img, factor):
    return np.clip(img * factor, 0.0, 1.0).astype(np.float32)

In [7]:
print('Extracting clean training features...')
train_feats = extract_batch(X_train_raw, 'train')
print('Done.')
for k, v in train_feats.items():
    print(f'  {k:10s}: {v.shape}')

with open(os.path.join(FEAT_DIR, 'train_feats.pkl'), 'wb') as f:
    pickle.dump({'features': train_feats, 'labels': y_train}, f)
print('Saved train_feats.pkl')

Extracting clean training features...
  train: 0/14034
  train: 1000/14034
  train: 2000/14034
  train: 3000/14034
  train: 4000/14034
  train: 5000/14034
  train: 6000/14034
  train: 7000/14034
  train: 8000/14034
  train: 9000/14034
  train: 10000/14034
  train: 11000/14034
  train: 12000/14034
  train: 13000/14034
  train: 14000/14034
Done.
  colour    : (14034, 96)
  hog       : (14034, 2916)
  lbp       : (14034, 26)
  spatial   : (14034, 2600)
Saved train_feats.pkl


In [6]:
test_feats = {}

# Clean images shared by both corruption types at level 0
print('Test: clean')
clean_feats = extract_batch(X_test_raw, 'test-clean')
test_feats['blur_0']       = clean_feats
test_feats['brightness_0'] = clean_feats

# Gaussian blur levels 1–4
for sigma in BLUR_SIGMAS[1:]:
    print(f'Test: blur sigma={sigma}')
    corrupted = np.array([apply_blur(img, sigma) for img in X_test_raw])
    test_feats[f'blur_{sigma}'] = extract_batch(corrupted, f'blur-{sigma}')

# Brightness levels 1–4
for level, factor in enumerate(BRIGHTNESS_FACTORS[1:], start=1):
    print(f'Test: brightness level={level} (factor={factor})')
    corrupted = np.array([apply_brightness(img, factor) for img in X_test_raw])
    test_feats[f'brightness_{level}'] = extract_batch(corrupted, f'bright-{factor}')

print('\nAll test features extracted:')
for key in test_feats:
    shapes = {k: v.shape for k, v in test_feats[key].items()}
    print(f'  {key}: {shapes}')

with open(os.path.join(FEAT_DIR, 'test_feats.pkl'), 'wb') as f:
    pickle.dump({'features': test_feats, 'labels': y_test}, f)
print('Saved test_feats.pkl')

Test: clean
  test-clean: 0/3000
  test-clean: 1000/3000
  test-clean: 2000/3000
Test: blur sigma=1
  blur-1: 0/3000
  blur-1: 1000/3000
  blur-1: 2000/3000
Test: blur sigma=2
  blur-2: 0/3000
  blur-2: 1000/3000
  blur-2: 2000/3000
Test: blur sigma=3
  blur-3: 0/3000
  blur-3: 1000/3000
  blur-3: 2000/3000
Test: blur sigma=4
  blur-4: 0/3000
  blur-4: 1000/3000
  blur-4: 2000/3000
Test: brightness level=1 (factor=0.8)
  bright-0.8: 0/3000
  bright-0.8: 1000/3000
  bright-0.8: 2000/3000
Test: brightness level=2 (factor=0.6)
  bright-0.6: 0/3000
  bright-0.6: 1000/3000
  bright-0.6: 2000/3000
Test: brightness level=3 (factor=0.4)
  bright-0.4: 0/3000
  bright-0.4: 1000/3000
  bright-0.4: 2000/3000
Test: brightness level=4 (factor=0.2)
  bright-0.2: 0/3000
  bright-0.2: 1000/3000
  bright-0.2: 2000/3000

All test features extracted:
  blur_0: {'colour': (3000, 96), 'hog': (3000, 2916), 'lbp': (3000, 26), 'spatial': (3000, 2600)}
  brightness_0: {'colour': (3000, 96), 'hog': (3000, 2916),